## <span style="color:purple">Morphological analysis with Bert based disambiguation</span>

The [estnltk_neural](https://github.com/estnltk/estnltk/tree/main/estnltk_neural) package contains `VabamorfWithBertTagger`, which provides Vabamorf's morphological analysis completed by _Bert based morphological disambiguation_. 
The latter is often more accurate than Vabamorf's morphological disambiguation, and can be additionally improved by adding an expert model for solving word form homonymy [(Saska 2026)](https://thesis.cs.ut.ee/8f10695d-4276-47ea-b306-5df66c034057). 

Under the hood, VabamorfWithBertTagger uses:

1. [VabamorfAnalyzer](01_morphological_analysis.ipynb) (for morph\_analysis layer) 
2. [PostMorphAnalysisTagger](01_morphological_analysis.ipynb) (for morph\_analysis post-corrections) 
3. [BertMorphTagger](08_bert_based_morph_tagger.ipynb) (as a disambiguator)
4. (optional) MorphHomonymsRetagger (for post-correcting Estonian homonymous word forms)

`VabamorfAnalyzer` will be applied with the default parameters `guess=True` and `propername=True`, but parameters `slang_lex`, `compound`, `phonetic`, `stem` can be changed (see [01_morphological_analysis.ipynb](01_morphological_analysis.ipynb) --> "Analysis parameters" for details about VabamorfAnalyzer's parameters). 

## Prerequisites

*Note: you need to install [estnltk_neural](https://github.com/estnltk/estnltk/tree/main/estnltk_neural) and [sentencepiece](https://pypi.org/project/sentencepiece/) packages for using the Bert-based morphological disambiguation models.*

Note that the Bert model required by the tagger is not distributed with the estnltk\_neural package. You can download the model in the following ways:
* If you create a new instance of `VabamorfWithBertTagger` and the model has not been downloaded yet, you'll be prompted with a question asking for a permission to download the model;
* Alternatively, you can pre-download the model manually via the download function:

```python
from estnltk import download
download('bert_morph_tagging')
```

## Examples how to use VabamorfWithBertTagger

### Default usage

In [1]:
from estnltk import Text
from estnltk_neural.taggers import VabamorfWithBertTagger

C:\Programmid\Miniconda3\envs\py313_devel\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
txt0 = Text("Meie ei pea otsima. Lahendused on leidnud meid ise!")
txt0.tag_layer(["words", "sentences"])
vbt = VabamorfWithBertTagger()
vbt.tag( txt0 )
txt0["morph_analysis"]

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 449.81it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Meie', [{'normalized_text': 'Meie', 'lemma': 'mina', 'root': 'mina', 'root_tokens': ['mina'], 'ending': '0', 'clitic': '', 'form': 'pl n', 'partofspeech': 'P'}]),
Span('ei', [{'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': 'neg', 'partofspeech': 'V'}]),
Span('pea', [{'normalized_text': 'pea', 'lemma': 'pidama', 'root': 'pida', 'root_tokens': ['pida'], 'ending': '0', 'clitic': '', 'form': 'neg o', 'partofspeech': 'V'}]),
Span('otsima', [{'normalized_text': 'otsima', 'lemma': 'otsima', 'root': 'otsi', 'root_tokens': ['otsi'], 'ending': 'ma', 'clitic': '', 'form': 'ma', 'partofspeech': 'V'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('Lahendused', [{'normalized_text': 'Lahendused', 'lemma': 'lahendus', 'root': 'lahendus', 'root_tokens': ['lahendus'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'S'}]),
Span('on', [{'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'vad', 'partofspeech': 'V'}]),
Span('leidnud', [{'normalized_text': 'leidnud', 'lemma': 'leidma', 'root': 'leid', 'root_tokens': ['leid'], 'ending': 'nud', 'clitic': '', 'form': 'nud', 'partofspeech': 'V'}]),
Span('meid', [{'normalized_text': 'meid', 'lemma': 'mina', 'root': 'mina', 'root_tokens': ['mina'], 'ending': 'd', 'clitic': '', 'form': 'pl p', 'partofspeech': 'P'}]),
Span('ise', [{'normalized_text': 'ise', 'lemma': 'ise', 'root': 'ise', 'root_tokens': ['ise'], 'ending': '0', 'clitic': '', 'form': 'pl n', 'partofspeech': 'P'}]),
Span('!', [{'normalized_text': '!', 'lemma': '!', 'root': '!', 'root_tokens': ['!'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

### Using hardware acceleration (GPU)

Use the parameter `device` to switch from CPU (default) to a GPU to speed up the processing, for example:

```python
from estnltk_neural.taggers import VabamorfWithBertTagger
morph_tagger = VabamorfWithBertTagger(device='cuda')  # use GPU
```

Note that this requires that you have the corresponding hardware available. By default, no hardware acceleration is used. 
For other possible `device` values, please consult [pytorch device documentation](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.device). 

### Verb corrections (flags correct_verb_annotation and change_to_bert_form)

Flags `correct_verb_annotation` and `change_to_bert_form` determine if verbs are updated based on BertMorpTagger analysis. By default, both flags are set to True.

When there is no verb multiplicity, then for 91% of the time (1789/1945 cases on UD treebank 2.18), the Bert's verb analysis improves the Vabamorf analysis. This is mostly because of "neg" in Bert's form analysis.

#### VabamorfWithBertTagger without verb corrections

In [3]:
vbt = VabamorfWithBertTagger(use_postanalysis=True,
                             correct_verb_annotation=False,
                             change_to_bert_form=False)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 351.55it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]


In [4]:
txt1 = Text("Meie ei pea otsima.")
txt1.tag_layer(["words", "sentences"])

vbt.tag( txt1 )
txt1["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Meie', [{'normalized_text': 'Meie', 'lemma': 'mina', 'root': 'mina', 'root_tokens': ['mina'], 'ending': '0', 'clitic': '', 'form': 'pl n', 'partofspeech': 'P'}]),
Span('ei', [{'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': 'neg', 'partofspeech': 'V'}]),
Span('pea', [{'normalized_text': 'pea', 'lemma': 'pea', 'root': 'pea', 'root_tokens': ['pea'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'pea', 'lemma': 'pea', 'root': 'pea', 'root_tokens': ['pea'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}, {'normalized_text': 'pea', 'lemma': 'pidama', 'root': 'pida', 'root_tokens': ['pida'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}, {'normalized_text': 'pea', 'lemma': 'pea', 'root': 'pea', 'root_tokens': ['pea'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('otsima', [{'normalized_text': 'otsima', 'lemma': 'otsima', 'root': 'otsi', 'root_tokens': ['otsi'], 'ending': 'ma', 'clitic': '', 'form': 'ma', 'partofspeech': 'V'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

#### VabamorfWithBertTagger with verb corrections (correcting only verb annotation)

In [5]:
vbt2 = VabamorfWithBertTagger(use_postanalysis=True,
                            correct_verb_annotation=True,
                            change_to_bert_form=False)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 330.57it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]


In [6]:
txt2 = Text("Meie ei pea otsima.")
txt2.tag_layer(["words", "sentences"])

vbt2.tag( txt2 )
txt2["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Meie', [{'normalized_text': 'Meie', 'lemma': 'mina', 'root': 'mina', 'root_tokens': ['mina'], 'ending': '0', 'clitic': '', 'form': 'pl n', 'partofspeech': 'P'}]),
Span('ei', [{'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': 'neg', 'partofspeech': 'V'}]),
Span('pea', [{'normalized_text': 'pea', 'lemma': 'pidama', 'root': 'pida', 'root_tokens': ['pida'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}]),
Span('otsima', [{'normalized_text': 'otsima', 'lemma': 'otsima', 'root': 'otsi', 'root_tokens': ['otsi'], 'ending': 'ma', 'clitic': '', 'form': 'ma', 'partofspeech': 'V'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

#### VabamorfWithBertTagger with verb corrections (correcting verb annotation and form)

In [7]:
vbt3 = VabamorfWithBertTagger(use_postanalysis=True,
                            correct_verb_annotation=True,
                            change_to_bert_form=True)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 403.38it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]


In [8]:
txt3 = Text("Meie ei pea otsima.")
txt3.tag_layer(["words", "sentences"])

vbt3.tag( txt3 )
txt3["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Meie', [{'normalized_text': 'Meie', 'lemma': 'mina', 'root': 'mina', 'root_tokens': ['mina'], 'ending': '0', 'clitic': '', 'form': 'pl n', 'partofspeech': 'P'}]),
Span('ei', [{'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': 'neg', 'partofspeech': 'V'}]),
Span('pea', [{'normalized_text': 'pea', 'lemma': 'pidama', 'root': 'pida', 'root_tokens': ['pida'], 'ending': '0', 'clitic': '', 'form': 'neg o', 'partofspeech': 'V'}]),
Span('otsima', [{'normalized_text': 'otsima', 'lemma': 'otsima', 'root': 'otsi', 'root_tokens': ['otsi'], 'ending': 'ma', 'clitic': '', 'form': 'ma', 'partofspeech': 'V'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

### VabamorfWithBertTagger: slang lex

By default, slang_lex flag is set to True. This helps to remove multiplicity for slang words.

In [9]:
# without slang lex
vbt4 = VabamorfWithBertTagger(use_postanalysis=True,
                            correct_verb_annotation=True,
                            change_to_bert_form=True,
                            slang_lex=False)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 406.26it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]


In [10]:
txt4 = Text("Ilm on mõnsa.")
txt4.tag_layer(["words", "sentences"])
vbt4.tag( txt4 )
txt4["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Ilm', [{'normalized_text': 'Ilm', 'lemma': 'ilm', 'root': 'ilm', 'root_tokens': ['ilm'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('on', [{'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'b', 'partofspeech': 'V'}]),
Span('mõnsa', [{'normalized_text': 'mõnsa', 'lemma': 'mõnsa', 'root': 'mõnsa', 'root_tokens': ['mõnsa'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}, {'normalized_text': 'mõnsa', 'lemma': 'mõnsa', 'root': 'mõnsa', 'root_tokens': ['mõnsa'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [11]:
# with slang lex
vbt5 = VabamorfWithBertTagger(use_postanalysis=True,
                            correct_verb_annotation=True,
                            change_to_bert_form=True,
                             slang_lex=True)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 350.14it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]


In [12]:
txt5 = Text("Ilm on mõnsa.")
txt5.tag_layer(["words", "sentences"])
vbt5.tag( txt5 )
txt5["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Ilm', [{'normalized_text': 'Ilm', 'lemma': 'ilm', 'root': 'ilm', 'root_tokens': ['ilm'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('on', [{'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'b', 'partofspeech': 'V'}]),
Span('mõnsa', [{'normalized_text': 'mõnsa', 'lemma': 'mõnsa', 'root': 'mõnsa', 'root_tokens': ['mõnsa'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

### VabamorfWithBertTagger with post-correction of homonymous word forms

`VabamorfWithBertTagger` accepts parameter `post_disambiguator`, which can be a retagger that refines / post-corrects Bert's predictions. 
While there is no post-correction by default, you can initialize the parameter with `MorphHomonymsRetagger` for post-correcting homonymous words. 

#### Prerequisites

Note that the Bert model required by `MorphHomonymsRetagger` is not distributed with the estnltk\_neural package. You can download the model in the following ways:
* If you create a new instance of `MorphHomonymsRetagger` and the model has not been downloaded yet, you'll be prompted with a question asking for a permission to download the model;
* Alternatively, you can pre-download the model manually via the download function:
```python
from estnltk import download
download('morph_homonymy_expert')
```

The default model for `MorphHomonymsRetagger` is [Ro-v2-H](https://huggingface.co/tartuNLP/est-roberta-vm-morph-homonym-tagging), which disambiguates homonymous words from inflectional types 1, 16, 17 and 19. 
The model creation and evaluation process are described in [(Saska 2026)](https://thesis.cs.ut.ee/8f10695d-4276-47ea-b306-5df66c034057).

In [13]:
from estnltk_neural.taggers import MorphHomonymsRetagger

In [14]:
# Use VabamorfWithBertTagger with homonymous word forms disambiguation expert model
vbt6 = VabamorfWithBertTagger(post_disambiguator=MorphHomonymsRetagger())

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 323.51it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]


In [15]:
txt6 = Text('Muna läks katki, sest kivi lõikas muna pooleks. Pärast viskas ta muna kiviga.')
txt6.tag_layer(["words", "sentences"])
vbt6.tag( txt6 )
txt6["morph_analysis"]

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Muna', [{'normalized_text': 'Muna', 'lemma': 'muna', 'root': 'muna', 'root_tokens': ['muna'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('läks', [{'normalized_text': 'läks', 'lemma': 'minema', 'root': 'mine', 'root_tokens': ['mine'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('katki', [{'normalized_text': 'katki', 'lemma': 'katki', 'root': 'katki', 'root_tokens': ['katki'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('sest', [{'normalized_text': 'sest', 'lemma': 'sest', 'root': 'sest', 'root_tokens': ['sest'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('kivi', [{'normalized_text': 'kivi', 'lemma': 'kivi', 'root': 'kivi', 'root_tokens': ['kivi'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('lõikas', [{'normalized_text': 'lõikas', 'lemma': 'lõikama', 'root': 'lõika', 'root_tokens': ['lõika'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('muna', [{'normalized_text': 'muna', 'lemma': 'muna', 'root': 'muna', 'root_tokens': ['muna'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}]),
Span('pooleks', [{'normalized_text': 'pooleks', 'lemma': 'pooleks', 'root': 'pooleks', 'root_tokens': ['pooleks'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('Pärast', [{'normalized_text': 'Pärast', 'lemma': 'pärast', 'root': 'pärast', 'root_tokens': ['pärast'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('viskas', [{'normalized_text': 'viskas', 'lemma': 'viskama', 'root': 'viska', 'root_tokens': ['viska'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('ta', [{'normalized_text': 'ta', 'lemma': 'tema', 'root': 'tema', 'root_tokens': ['tema'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span('muna', [{'normalized_text': 'muna', 'lemma': 'muna', 'root': 'muna', 'root_tokens': ['muna'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('kiviga', [{'normalized_text': 'kiviga', 'lemma': 'kivi', 'root': 'kivi', 'root_tokens': ['kivi'], 'ending': 'ga', 'clitic': '', 'form': 'sg kom', 'partofspeech': 'S'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

**Hardware acceleration (GPU)**. If you want to use a GPU to speed up the processing, then initialize MorphHomonymsRetagger with  the parameter `device` set to 'cuda':
```python
from estnltk_neural.taggers import MorphHomonymsRetagger
expert_model = MorphHomonymsRetagger(device='cuda')  # use GPU
```